# B3a：公司行動 v2 衍生資料集（離線，不需網路）

只讀官方參考資料 `reference/corporate_actions/v1` 與 lake 的 FinLab 日 K，產生新的不可變衍生資料夾：
- `corporate_actions_v2.parquet`：三個日期分開（官方生效日／生效交易時段／該檔第一個成交日）、價格因子、會計狀態 SUPPORTED／PRICE_ONLY／UNSUPPORTED、retrieved_at、available_at（08:30 為政策）
- `diagnostics_ledger.parquet`：每個事件 FinLab 何時調整
- `finlab_jumps_without_event.csv`、`research_quarantine.csv`（8101）、`manifest.json`（SHA-256）

不修改原始資料、manifest、發布指標、遮罩。**依序執行全部格子即可**，約 3–8 分鐘。

In [ ]:
# 1. 參數
REPO_OWNER, REPO_NAME, BRANCH = "hh4832", "tw-market-data-lake", "main"
DATA_ROOT = "/content/drive/MyDrive/00Quant_Research/data"
LAKE = f"{DATA_ROOT}/_tw-market-data-lake"
CA_ROOT = f"{LAKE}/reference/corporate_actions/v1"
OFFICIAL_ROOT = f"{LAKE}/reference/official_volume/v1"
GAP_LEDGER = f"{LAKE}/audits/gap_ledger_20261026_163358_3e8eed0/gap_day_summary.parquet"
PROJECT_DIR = f"/content/{REPO_NAME}"
OUT_DIR = "/content/tw_lake_ca_v2"
ARCHIVE = f"{LAKE}/derived/corporate_actions_v2"
PREFIX = "corporate_actions_v2"


In [ ]:
# 2. 掛載 Google Drive
from google.colab import drive
drive.mount("/content/drive")


In [ ]:
# 3. clone / pull + 依賴
import subprocess, os
if os.path.exists(f"{PROJECT_DIR}/.git"):
    subprocess.run(["git", "-C", PROJECT_DIR, "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", PROJECT_DIR, "checkout", "-B", BRANCH, f"origin/{BRANCH}"], check=True)
else:
    subprocess.run(["git", "clone", "--branch", BRANCH, f"https://github.com/{REPO_OWNER}/{REPO_NAME}.git", PROJECT_DIR], check=True)
print("HEAD:", subprocess.check_output(["git", "-C", PROJECT_DIR, "rev-parse", "HEAD"], text=True).strip())
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml>=6", "tqdm>=4.66"], check=True)
def stream(args):
    env = dict(os.environ, PYTHONPATH=f"{PROJECT_DIR}/src", PYTHONUNBUFFERED="1")
    p = subprocess.Popen(args, cwd=PROJECT_DIR, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end="")
    return p.wait()

def ensure_drive():
    """Mount Drive safely. If an earlier cell unmounted Drive and then wrote under /content/drive,
    those files are only on this runtime's disk: move them aside (never delete) and copy any audit
    folders that are missing on the real Drive."""
    import os, shutil, time
    from google.colab import drive
    if os.path.ismount("/content/drive"):
        return
    stray = None
    if os.path.isdir("/content/drive") and os.listdir("/content/drive"):
        stray = f"/content/drive_local_{int(time.time())}"
        shutil.move("/content/drive", stray)
        print("moved local (unsynced) files to", stray)
    drive.mount("/content/drive")
    if stray:
        src_root = f"{stray}/MyDrive/00Quant_Research/data/_tw-market-data-lake/audits"
        dst_root = f"{DATA_ROOT}/_tw-market-data-lake/audits"
        for dirpath, dirnames, filenames in os.walk(src_root):
            rel = os.path.relpath(dirpath, src_root)
            for d in list(dirnames):
                target = os.path.join(dst_root, rel, d)
                if not os.path.exists(target):
                    shutil.copytree(os.path.join(dirpath, d), target)
                    print("copied missing", target)
                    dirnames.remove(d)

In [ ]:
# 4. 執行
print("exit", stream([sys.executable, "-m", "tw_lake.audit.corporate_actions_v2", "--dataset-root", CA_ROOT,
                      "--data-root", DATA_ROOT, "--out-root", OUT_DIR]))


In [ ]:
# 5. 重點結果，並複製到 Drive（把這格輸出貼給 Claude）
import glob, json, os, shutil
import pandas as pd
ensure_drive()
pd.set_option("display.width", 250); pd.set_option("display.max_rows", 300)
d = sorted(glob.glob(f"{OUT_DIR}/{PREFIX}_*"))[-1]
s = json.load(open(f"{d}/summary.json", encoding="utf-8"))
print(json.dumps({k: v for k, v in s.items() if k not in ("accounting_status", "timing_status_by_type")}, ensure_ascii=False, indent=1))
print("\n== accounting status"); print(pd.DataFrame(s["accounting_status"]).to_string(index=False))
print("\n== timing by type"); print(pd.DataFrame(s.get("timing_status_by_type", [])).pivot_table(index="event_type", columns="timing_status", values="n", fill_value=0).to_string())
dg = pd.read_parquet(f"{d}/diagnostics_ledger.parquet")
x = dg[dg["timing_status"].isin(["UNEXPLAINED_TIMING", "MATCH_FIRST_OBSERVED_TRADE"])]
print("\n== non-session timing cases"); print(x[["symbol", "event_type", "official_effective_date", "effective_market_session", "first_observed_trade_date", "finlab_jump_date", "factor_rel_err", "timing_status"]].to_string(index=False))
print("\n== FinLab jumps without event"); print(pd.read_csv(f"{d}/finlab_jumps_without_event.csv", dtype={"symbol": str}).to_string(index=False))
print("\n== manifest"); print(open(f"{d}/manifest.json", encoding="utf-8").read())
os.makedirs(ARCHIVE, exist_ok=True)
dst = f"{ARCHIVE}/{os.path.basename(d)}"
if not os.path.exists(dst):
    shutil.copytree(d, dst)
print("\ncopied to", dst)
